# 01 Data Cleaning and Integration

This notebook prepares the interim and integrated datasets for the capstone project:

**A Data-Driven Framework for Prioritizing Public Capital Allocation to Basic Education Infrastructure across Nigerian States Using Education Need Indicators**

## Purpose

This notebook performs the following tasks:

1. Loads the project workbook.
2. Reads the core and expanded model-ready sheets.
3. Creates cleaned interim files for UBEC, NBS, Budget/Allocation context, and DMO context.
4. Creates the state-name mapping file.
5. Creates the merged state-year interim dataset.
6. Creates the final integrated processed state-year dataset.

## Source Workbook

Expected location:

```text
../data/raw/Capstone_Stage_ML1_Model_Ready_Dataset.xlsx
```

If the workbook is stored elsewhere, update the `WORKBOOK_PATH` variable below.


In [ ]:
# Import required libraries.
from pathlib import Path
import pandas as pd
import numpy as np

# Configure pandas display options for easier inspection.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Define project paths.
PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create output directories if they do not already exist.
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Define source workbook path.
WORKBOOK_PATH = RAW_DIR / "Capstone_Stage_ML1_Model_Ready_Dataset.xlsx"

# Confirm expected source file path.
WORKBOOK_PATH


## 1. Load Workbook Sheets

The workbook contains the project’s final model-ready sheets and supporting documentation sheets.


In [ ]:
# Load the Excel workbook.
excel_file = pd.ExcelFile(WORKBOOK_PATH)

# Display available sheet names.
excel_file.sheet_names


In [ ]:
# Load the core BENI state-year sheet.
core_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Model_Ready_Core_StateYear")

# Load the expanded 2022 BENI sheet.
expanded_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Model_Ready_Expanded_2022")

# Load the feature dictionary sheet.
feature_dictionary_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Feature_Dictionary")

# Confirm dataset shapes.
print("Core dataset shape:", core_df.shape)
print("Expanded dataset shape:", expanded_df.shape)
print("Feature dictionary shape:", feature_dictionary_df.shape)


## 2. Basic Quality Checks

These checks confirm state coverage, year coverage, and missingness before exporting interim and processed files.


In [ ]:
# Check core dataset coverage.
print("Core unique states:", core_df["state"].nunique())
print("Core years:", sorted(core_df["year"].dropna().unique().tolist()))
print("Core missing values:", int(core_df.isna().sum().sum()))

# Check expanded dataset coverage.
print("Expanded unique states:", expanded_df["state"].nunique())
print("Expanded years:", sorted(expanded_df["year"].dropna().unique().tolist()))
print("Expanded missing values:", int(expanded_df.isna().sum().sum()))


## 3. Create `ubec_cleaned_interim.csv`

This file preserves the cleaned UBEC-derived education infrastructure indicators used in BENI-Core.


In [ ]:
# Select UBEC-derived education infrastructure fields from the core dataset.
ubec_columns = [
    "state",
    "year",
    "target_beni_core_score",
    "beni_rank_within_year",
    "priority_band_within_year",
    "pupil_teacher_ratio",
    "pupil_classroom_ratio",
    "unqualified_teacher_share",
    "bad_classroom_share",
    "enrolment_per_school",
    "matching_grant_million_naira",
    "core_indicator_completeness",
    "split_group",
    "model_ready_flag",
    "source_note",
]

# Create UBEC cleaned interim dataframe.
ubec_cleaned_interim = core_df[ubec_columns].copy()

# Add audit metadata.
ubec_cleaned_interim["source_file"] = "Capstone_Stage_ML1_Model_Ready_Dataset.xlsx"
ubec_cleaned_interim["source_sheet"] = "Model_Ready_Core_StateYear"
ubec_cleaned_interim["cleaning_status"] = "cleaned_interim"
ubec_cleaned_interim["interim_note"] = "UBEC-derived education infrastructure indicators retained for BENI-Core analysis."

# Export file.
ubec_cleaned_interim.to_csv(INTERIM_DIR / "ubec_cleaned_interim.csv", index=False)

# Display preview.
ubec_cleaned_interim.head()


## 4. Create `nbs_cleaned_interim.csv`

This file preserves the NBS-derived socioeconomic need proxy and related population field used in BENI-Expanded.


In [ ]:
# Select NBS-related fields from the expanded dataset.
nbs_columns = [
    "state",
    "year",
    "nbs_socioeconomic_need_proxy",
    "nbs_socioeconomic_need_proxy_norm",
    "school_age_basic_population",
    "split_group",
    "model_ready_flag",
    "source_note",
]

# Create NBS cleaned interim dataframe.
nbs_cleaned_interim = expanded_df[nbs_columns].copy()

# Add audit metadata.
nbs_cleaned_interim["source_file"] = "Capstone_Stage_ML1_Model_Ready_Dataset.xlsx"
nbs_cleaned_interim["source_sheet"] = "Model_Ready_Expanded_2022"
nbs_cleaned_interim["cleaning_status"] = "cleaned_interim"
nbs_cleaned_interim["interim_note"] = "NBS-derived socioeconomic need proxy retained for BENI-Expanded analysis."

# Export file.
nbs_cleaned_interim.to_csv(INTERIM_DIR / "nbs_cleaned_interim.csv", index=False)

# Display preview.
nbs_cleaned_interim.head()


## 5. Create Budget/Allocation Context File

The available allocation-context field in the workbook is `matching_grant_million_naira`. This is retained as contextual evidence and not treated as a direct BENI input.


In [ ]:
# Create budget/allocation context file from the available matching-grant variable.
budget_cleaned_interim = core_df[
    [
        "state",
        "year",
        "matching_grant_million_naira",
        "split_group",
        "model_ready_flag",
        "source_note",
    ]
].copy()

# Add derived naira equivalent.
budget_cleaned_interim["budget_context_variable"] = "matching_grant_million_naira"
budget_cleaned_interim["amount_naira"] = budget_cleaned_interim["matching_grant_million_naira"] * 1_000_000

# Add audit metadata.
budget_cleaned_interim["source_file"] = "Capstone_Stage_ML1_Model_Ready_Dataset.xlsx"
budget_cleaned_interim["source_sheet"] = "Model_Ready_Core_StateYear"
budget_cleaned_interim["cleaning_status"] = "cleaned_interim"
budget_cleaned_interim["interim_note"] = "Matching grant retained as available allocation-context evidence."

# Reorder columns.
budget_cleaned_interim = budget_cleaned_interim[
    [
        "state",
        "year",
        "budget_context_variable",
        "matching_grant_million_naira",
        "amount_naira",
        "split_group",
        "model_ready_flag",
        "source_file",
        "source_sheet",
        "source_note",
        "cleaning_status",
        "interim_note",
    ]
]

# Export file.
budget_cleaned_interim.to_csv(INTERIM_DIR / "budget_cleaned_interim.csv", index=False)

# Display preview.
budget_cleaned_interim.head()


## 6. Create DMO Fiscal-Context File

DMO debt variables are retained for fiscal-context interpretation and are not direct BENI inputs.


In [ ]:
# Select DMO-related fiscal-context fields.
dmo_cleaned_interim = expanded_df[
    [
        "state",
        "year",
        "dmo_domestic_debt_stock_naira",
        "fiscal_pressure_band",
        "split_group",
        "model_ready_flag",
        "source_note",
    ]
].copy()

# Add period and metadata.
dmo_cleaned_interim["period"] = "2025_Q1_or_source_period"
dmo_cleaned_interim["source_file"] = "Capstone_Stage_ML1_Model_Ready_Dataset.xlsx"
dmo_cleaned_interim["source_sheet"] = "Model_Ready_Expanded_2022"
dmo_cleaned_interim["cleaning_status"] = "cleaned_interim"
dmo_cleaned_interim["interim_note"] = "DMO domestic debt retained as fiscal-context evidence only."

# Reorder columns.
dmo_cleaned_interim = dmo_cleaned_interim[
    [
        "state",
        "year",
        "period",
        "dmo_domestic_debt_stock_naira",
        "fiscal_pressure_band",
        "split_group",
        "model_ready_flag",
        "source_file",
        "source_sheet",
        "source_note",
        "cleaning_status",
        "interim_note",
    ]
]

# Export file.
dmo_cleaned_interim.to_csv(INTERIM_DIR / "dmo_cleaned_interim.csv", index=False)

# Display preview.
dmo_cleaned_interim.head()


## 7. Create `state_name_mapping.csv`

This file documents state-name harmonisation across sheets.


In [ ]:
# Collect state labels from all workbook sheets containing a state column.
state_records = []

# Loop through all workbook sheets.
for sheet in excel_file.sheet_names:
    sheet_df = pd.read_excel(WORKBOOK_PATH, sheet_name=sheet)

    # Identify possible state columns.
    possible_state_cols = [
        col for col in sheet_df.columns 
        if str(col).strip().lower() in ["state", "state_name", "states"]
    ]

    # Extract state labels.
    for col in possible_state_cols:
        states = sheet_df[col].dropna().astype(str).str.strip().unique()
        for raw_state in states:
            state_records.append({
                "raw_state_name": raw_state,
                "source_sheet": sheet
            })

# Create mapping dataframe.
state_mapping_df = pd.DataFrame(state_records).drop_duplicates()

# Define state standardisation function.
def standardize_state_name(name):
    cleaned = str(name).strip()
    variants = {
        "Federal Capital Territory": "FCT",
        "F.C.T": "FCT",
        "F.C.T.": "FCT",
        "Abuja": "FCT",
        "FCT Abuja": "FCT",
        "Nassarawa": "Nasarawa",
    }
    return variants.get(cleaned, cleaned)

# Apply standardisation.
state_mapping_df["standard_state_name"] = state_mapping_df["raw_state_name"].apply(standardize_state_name)

# Add notes.
state_mapping_df["notes"] = np.where(
    state_mapping_df["raw_state_name"] == state_mapping_df["standard_state_name"],
    "No change",
    "Standardised to " + state_mapping_df["standard_state_name"]
)

# Reorder and sort.
state_mapping_df = state_mapping_df[
    ["raw_state_name", "standard_state_name", "source_sheet", "notes"]
].sort_values(["standard_state_name", "raw_state_name", "source_sheet"])

# Export file.
state_mapping_df.to_csv(INTERIM_DIR / "state_name_mapping.csv", index=False)

# Display summary.
print("State mapping rows:", len(state_mapping_df))
print("Unique standardized states:", state_mapping_df["standard_state_name"].nunique())
state_mapping_df.head()


## 8. Create `merged_state_year_interim.csv`

This file keeps all 148 core state-year records and joins expanded 2022 NBS/DMO/context fields where available.


In [ ]:
# Select core fields for interim merging.
core_merge_columns = [
    "state",
    "year",
    "target_beni_core_score",
    "beni_rank_within_year",
    "priority_band_within_year",
    "pupil_teacher_ratio",
    "pupil_classroom_ratio",
    "unqualified_teacher_share",
    "bad_classroom_share",
    "enrolment_per_school",
    "matching_grant_million_naira",
    "core_indicator_completeness",
    "split_group",
    "model_ready_flag",
    "source_note",
]

# Select expanded/context fields for interim merging.
expanded_merge_columns = [
    "state",
    "year",
    "target_beni_expanded_score",
    "beni_expanded_rank",
    "nbs_socioeconomic_need_proxy",
    "school_age_basic_population",
    "dmo_domestic_debt_stock_naira",
    "fiscal_pressure_band",
    "matching_grant_2018_2022_naira",
    "cv_fold",
]

# Merge expanded fields into core state-year framework.
merged_state_year_interim = core_df[core_merge_columns].merge(
    expanded_df[expanded_merge_columns],
    on=["state", "year"],
    how="left",
    validate="many_to_one"
)

# Add metadata.
merged_state_year_interim["source_file"] = "Capstone_Stage_ML1_Model_Ready_Dataset.xlsx"
merged_state_year_interim["source_sheets"] = "Model_Ready_Core_StateYear; Model_Ready_Expanded_2022"
merged_state_year_interim["merge_status"] = np.where(
    merged_state_year_interim["target_beni_expanded_score"].notna(),
    "merged_with_expanded_2022_context",
    "core_state_year_only"
)
merged_state_year_interim["cleaning_status"] = "interim_merged"
merged_state_year_interim["interim_note"] = (
    "Core UBEC-derived rows retained for all available years; "
    "expanded NBS/DMO/context fields joined where available for 2022."
)

# Export file.
merged_state_year_interim.to_csv(INTERIM_DIR / "merged_state_year_interim.csv", index=False)

# Display merge summary.
print("Merged shape:", merged_state_year_interim.shape)
print(merged_state_year_interim["merge_status"].value_counts())
merged_state_year_interim.head()


## 9. Create Final Integrated Processed Dataset

This output is saved to `data/processed/integrated_state_year_dataset.csv`.


In [ ]:
# Merge all expanded 2022 fields into the core state-year framework.
integrated_state_year_dataset = core_df.merge(
    expanded_df,
    on=["state", "year"],
    how="left",
    suffixes=("_core", "_expanded"),
    validate="many_to_one"
)

# Add processed metadata.
integrated_state_year_dataset["processed_source_file"] = "Capstone_Stage_ML1_Model_Ready_Dataset.xlsx"
integrated_state_year_dataset["processed_source_sheets"] = "Model_Ready_Core_StateYear; Model_Ready_Expanded_2022"
integrated_state_year_dataset["processed_dataset_name"] = "integrated_state_year_dataset"
integrated_state_year_dataset["processed_status"] = "final_integrated_dataset"
integrated_state_year_dataset["integration_note"] = (
    "Core state-year records retained for 2018, 2019, 2020, and 2022. "
    "Expanded 2022 NBS, DMO, and BENI-Expanded fields joined where available."
)

# Export integrated dataset.
integrated_state_year_dataset.to_csv(
    PROCESSED_DIR / "integrated_state_year_dataset.csv",
    index=False
)

# Display processed dataset summary.
print("Integrated dataset shape:", integrated_state_year_dataset.shape)
print("Rows with expanded context:", integrated_state_year_dataset["target_beni_expanded_score"].notna().sum())
print("Core-only rows:", integrated_state_year_dataset["target_beni_expanded_score"].isna().sum())
integrated_state_year_dataset.head()


## 10. Notebook Output Summary

At the end of this notebook, the following files should exist:

```text
data/interim/ubec_cleaned_interim.csv
data/interim/nbs_cleaned_interim.csv
data/interim/budget_cleaned_interim.csv
data/interim/dmo_cleaned_interim.csv
data/interim/state_name_mapping.csv
data/interim/merged_state_year_interim.csv
data/processed/integrated_state_year_dataset.csv
```


In [ ]:
# Confirm exported files.
expected_files = [
    INTERIM_DIR / "ubec_cleaned_interim.csv",
    INTERIM_DIR / "nbs_cleaned_interim.csv",
    INTERIM_DIR / "budget_cleaned_interim.csv",
    INTERIM_DIR / "dmo_cleaned_interim.csv",
    INTERIM_DIR / "state_name_mapping.csv",
    INTERIM_DIR / "merged_state_year_interim.csv",
    PROCESSED_DIR / "integrated_state_year_dataset.csv",
]

# Print file-existence status.
for file_path in expected_files:
    print(f"{file_path}: {'FOUND' if file_path.exists() else 'MISSING'}")
